In [ ]:
pip install streamlit openai tavily-python python-dotenv

In [ ]:
OPENAI_API_KEY= "live-ai-assistant"
TAVILY_API_KEY= "tavily_key"

In [ ]:
%%writefile app.py
import streamlit as st
import os
from openai import OpenAI
from tavily import TavilyClient

# Google Colab mein keys access karne ka tareeqa (Upar diye gaye cell k mutabiq)
# Agar aap ne Colab Secrets (🔑 icon) use kiya hai to aise read hoga:
try:
    from google.colab import userdata
    openai_key = userdata.get('OPENAI_API_KEY')
    tavily_key = userdata.get('TAVILY_API_KEY')
except:
    # Agar simple environment variables hain
    openai_key = os.getenv("OPENAI_API_KEY")
    tavily_key = os.getenv("TAVILY_API_KEY")

# APIs Initialize karein
openai_client = OpenAI(api_key=openai_key)
tavily_client = TavilyClient(api_key=tavily_key)

# 1. Web Search Tool Function
def web_search(query: str):
    """Internet se latest information search krny k liye tool."""
    try:
        response = tavily_client.search(query=query, max_results=3)
        results = [f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}" for r in response['results']]
        return "\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

# 2. Main Agent Function (With Memory)
def run_assistant(user_prompt, chat_history):
    messages = [
        {"role": "system", "content": "You are a Live AI Assistant with real-time internet access. Always verify facts using the web_search tool for recent events or real-time data before answering."}
    ]
    
    for interaction in chat_history:
        messages.append({"role": "user", "content": interaction["user"]})
        messages.append({"role": "assistant", "content": interaction["assistant"]})
        
    messages.append({"role": "user", "content": user_prompt})
    
    tools = [{
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the live internet for recent news, facts, and real-time data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query to look up on the internet."}
                },
                "required": ["query"]
            }
        }
    }]
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    if tool_calls:
        messages.append(response_message)
        for tool_call in tool_calls:
            if tool_call.function.name == "web_search":
                import json
                args = json.loads(tool_call.function.arguments)
                search_result = web_search(query=args['query'])
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": "web_search",
                    "content": search_result
                })
        
        final_response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return final_response.choices[0].message.content
    
    return response_message.content

# 3. Streamlit UI Setup
st.set_page_config(page_title="Live AI Assistant", layout="centered")
st.title("🌐 Live AI Assistant")
st.caption("Real-time Question Answering, Fact Verification & Internet Search")

if "memory" not in st.session_state:
    st.session_state.memory = []

for chat in st.session_state.memory:
    with st.chat_message("user"):
        st.write(chat["user"])
    with st.chat_message("assistant"):
        st.write(chat["assistant"])

if user_input := st.chat_input("Ask me anything..."):
    with st.chat_message("user"):
        st.write(user_input)
        
    with st.chat_message("assistant"):
        with st.spinner("Searching and thinking..."):
            answer = run_assistant(user_input, st.session_state.memory)
            st.write(answer)
            
    st.session_state.memory.append({"user": user_input, "assistant": answer})

In [ ]:
!wget -qO- ipv4.icanhazip.com

In [ ]:
!npm install -g localtunnel

In [ ]:
!pip install streamlit openai tavily-python python-dotenv && python -m streamlit run app.py & npx localtunnel --port 8501

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501     